#### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


In [4]:
from langchain.chat_models import init_chat_model

model=init_chat_model("gpt-5-nano")
model

ChatOpenAI(output_version=None, profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001852BA69310>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001852BA696D0>, root_client=<openai.OpenAI object at 0x000001852BA69090>, root_async_client=<openai.AsyncOpenAI object at 0x000001852BA69450>, model_name='gpt-5-nano', model_kwargs={}, o

In [8]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather of a location"""
    return f"Its sunny in {location}"

model_with_tools=model.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatOpenAI(output_version=None, profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001852BA69310>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001852BA696D0>, root_client=<openai.OpenAI object at 0x000001852BA69090>, root_async_client=<openai.AsyncOpenAI object at 0x000001852BA69450>, model_name='gpt-5-n

In [12]:
response = model_with_tools.invoke("What's the weather in boston")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 131, 'total_tokens': 282, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Do4TkiR0reoFsOePQeXcnZUJmVHVJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ea180-1a9d-77a3-ba62-259b8fcecf04-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_AlT00422TEAqqp57bWQ8Xup0', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 131, 'output_tokens': 151, 'total_tokens': 282, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}}
Tool: get_weather
Args: {'loc

#### Tool Execution Loops

In [15]:
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
print(ai_msg)
messages.append(ai_msg)
print(messages)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    print(tool_result)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)

print(messages)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 217, 'prompt_tokens': 131, 'total_tokens': 348, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Do4r8w9xcyGnEt9zBPRVn1974tnIU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ea196-3a75-7471-a103-ca6aa0642131-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston, MA'}, 'id': 'call_PkiWZLdq2KHTby6UNaCa640V', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 131, 'output_tokens': 217, 'total_tokens': 348, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}}
[{'role': 'user', 'conten